# M9 DLNLP - ASSIGNMENT 2
## Attention Mechanism

Business Context:

In Natural Language Processing, understanding relationships between words in a sentence is crucial. The attention mechanism helps models identify which words are more important while processing a given word.

In this assignment, we implement a basic self-attention mechanism using PyTorch. The model computes attention scores between words and is trained to learn a meaningful relationship — specifically, how a pronoun like “he” refers to a subject like “Elon”.

**Scenario**

Consider the following sentence:

_"Elon Musk built a rocket and he celebrated"_

The goal of this assignment is to train a simple attention mechanism so that the word "he" attends strongly to the word "Elon".

Tasks

1. Sentence Preparation

Create a Python list containing the words from the sentence: "Elon Musk built a rocket and he celebrated"
Also display the total number of words present in the sentence.

2. Creating Word Embeddings

Generate random embeddings for each word using PyTorch. Use an embedding dimension of 4. Display the embedding matrix and explain its shape. Create random weight matrices for Query (Wq) and Key (Wk).

3. Computing Query and Key Matrices

Compute the Query (Q) and Key (K) matrices using matrix multiplication. Display the output matrices and explain their role in attention computation. Compute the attention scores using the formula: Attention Score = (Q × Kᵀ) / √d where d is the embedding dimension. Apply the softmax function to convert scores into attention probabilities.

4. Attention Analysis Before Training

Print the attention values corresponding to the word "he". Identify which word receives the highest attention before training and comment on whether the attention appears meaningful.

5. Training the Attention Mechanism

Train the attention mechanism using gradient descent so that the word "he" learns to attend strongly to the word "Elon".

Use:
- Cross entropy loss
- Learning rate = 0.5
- Number of epochs = 200

Explain the purpose of the target index used during training.

6. Attention Analysis After Training

After training, recompute the attention matrix. Print the new attention values for the word "he". Compare the results before and after training.

## Import Libraries

This assignment follows the same self-attention approach used in the "M9 Unit 6 Attention Mechanisms" class notebook (Query/Key only, no Value), adapted to the sentence "Elon Musk built a rocket and he celebrated" so that "he" learns to attend to "Elon".

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn.functional as F

## Task 1: Sentence Preparation

In [2]:
sentence = "Elon Musk built a rocket and he celebrated"
words = sentence.split()

print(f'Number of words: {len(words)}')
words

Number of words: 8


['Elon', 'Musk', 'built', 'a', 'rocket', 'and', 'he', 'celebrated']

## Task 2: Creating Word Embeddings

Each word is represented by a random 4-dimensional embedding. `Wq` and `Wk` are the trainable weight matrices that will later project embeddings into query and key vectors.

In [3]:
torch.manual_seed(0)

embedding_dim = 4
X = torch.rand((len(words), embedding_dim))

print(f'Embedding matrix shape: {tuple(X.shape)}  # (num_words={len(words)}, embedding_dim={embedding_dim})')
X

Embedding matrix shape: (8, 4)  # (num_words=8, embedding_dim=4)


tensor([[0.4963, 0.7682, 0.0885, 0.1320],
        [0.3074, 0.6341, 0.4901, 0.8964],
        [0.4556, 0.6323, 0.3489, 0.4017],
        [0.0223, 0.1689, 0.2939, 0.5185],
        [0.6977, 0.8000, 0.1610, 0.2823],
        [0.6816, 0.9152, 0.3971, 0.8742],
        [0.4194, 0.5529, 0.9527, 0.0362],
        [0.1852, 0.3734, 0.3051, 0.9320]])

In [4]:
Wq = torch.rand((embedding_dim, embedding_dim), requires_grad=True)
Wk = torch.rand((embedding_dim, embedding_dim), requires_grad=True)

## Task 3: Computing Query and Key Matrices

`Q` represents what each word is looking for, `K` represents what each word offers. Matching queries against keys produces the attention scores.

In [5]:
Q = X @ Wq
K = X @ Wk

print('Q shape:', tuple(Q.shape))
print('K shape:', tuple(K.shape))

Q shape: (8, 4)
K shape: (8, 4)


In [6]:
d_k = K.shape[-1]

# Scaled dot-product attention scores, then softmax to get a probability distribution per word
scores = Q @ K.T / np.sqrt(d_k)
attention_before = F.softmax(scores, dim=-1)
attention_before

tensor([[0.0987, 0.1472, 0.1126, 0.0566, 0.1392, 0.2477, 0.0932, 0.1048],
        [0.0812, 0.1456, 0.0986, 0.0314, 0.1412, 0.3437, 0.0756, 0.0828],
        [0.0934, 0.1478, 0.1086, 0.0469, 0.1405, 0.2770, 0.0884, 0.0973],
        [0.1095, 0.1400, 0.1188, 0.0712, 0.1397, 0.2054, 0.1065, 0.1088],
        [0.0918, 0.1489, 0.1076, 0.0460, 0.1397, 0.2825, 0.0859, 0.0977],
        [0.0710, 0.1443, 0.0898, 0.0235, 0.1365, 0.3956, 0.0647, 0.0746],
        [0.0939, 0.1502, 0.1092, 0.0490, 0.1372, 0.2698, 0.0917, 0.0989],
        [0.0932, 0.1445, 0.1079, 0.0436, 0.1442, 0.2857, 0.0879, 0.0929]],
       grad_fn=<SoftmaxBackward0>)

## Task 4: Attention Analysis Before Training

In [7]:
he_index = words.index('he')
elon_index = words.index('Elon')

print("Attention of 'he' BEFORE training:\n")
for word, score in zip(words, attention_before[he_index]):
    print(f'{word:>10}: {score.item():.4f}')

top_word = words[attention_before[he_index].argmax().item()]
print(f"\nWord receiving highest attention from 'he': {top_word}")

Attention of 'he' BEFORE training:

      Elon: 0.0939
      Musk: 0.1502
     built: 0.1092
         a: 0.0490
    rocket: 0.1372
       and: 0.2698
        he: 0.0917
celebrated: 0.0989

Word receiving highest attention from 'he': and


Before training, `Wq` and `Wk` are random, so the attention scores for "he" are close to uniform across the sentence and do not reflect any real linguistic relationship — whichever word happens to score highest here is coincidental, not a learned reference to "Elon".

## Task 5: Training the Attention Mechanism

`target_index` is the supervised label (the position of "Elon") that the cross-entropy loss uses to push the attention distribution of "he" towards that word — it plays the role of the correct answer in a pronoun-resolution task, and the loop below updates `Wq`/`Wk` by gradient descent so the model's predicted attention gets closer to it on every epoch.

In [8]:
target_index = torch.tensor([elon_index])
learning_rate = 0.5
epochs = 200

for epoch in range(epochs):
    Q = X @ Wq
    K = X @ Wk
    scores = Q @ K.T / np.sqrt(d_k)
    attention_weights = F.softmax(scores, dim=-1)

    predicted = attention_weights[he_index].unsqueeze(0)
    loss = F.cross_entropy(predicted, target_index)

    loss.backward()
    with torch.no_grad():
        Wq -= learning_rate * Wq.grad
        Wk -= learning_rate * Wk.grad
    Wq.grad.zero_()
    Wk.grad.zero_()

    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch + 1:>3}/{epochs} - loss: {loss.item():.4f}')

Epoch  50/200 - loss: 2.0330
Epoch 100/200 - loss: 1.6058
Epoch 150/200 - loss: 1.3363
Epoch 200/200 - loss: 1.2987


## Task 6: Attention Analysis After Training

In [9]:
Q = X @ Wq
K = X @ Wk
scores = Q @ K.T / np.sqrt(d_k)
attention_after = F.softmax(scores, dim=-1)

print("Attention of 'he' AFTER training:\n")
for word, score in zip(words, attention_after[he_index]):
    print(f'{word:>10}: {score.item():.4f}')

Attention of 'he' AFTER training:

      Elon: 0.9706
      Musk: 0.0000
     built: 0.0015
         a: 0.0010
    rocket: 0.0266
       and: 0.0000
        he: 0.0004
celebrated: 0.0000


In [10]:
import pandas as pd

comparison = pd.DataFrame({
    'Word': words,
    'Attention Before': attention_before[he_index].detach().numpy().round(4),
    'Attention After': attention_after[he_index].detach().numpy().round(4),
})
comparison

,Word,Attention Before,Attention After
0,Elon,0.0939,0.9706
1,Musk,0.1502,0.0000
2,built,0.1092,0.0015
3,a,0.0490,0.0010
4,rocket,0.1372,0.0266
5,and,0.2698,0.0000
6,he,0.0917,0.0004
7,celebrated,0.0989,0.0000


## Conclusion

Before training, the attention weights for "he" were close to uniform across the sentence, reflecting the fact that `Wq`/`Wk` were still random and had learned nothing about the sentence's structure. After 200 epochs of gradient descent on the cross-entropy loss, "he" attends to "Elon" with probability close to 0.97, while the loss steadily decreases across epochs.

This shows that training only the Query/Key projections (no Value matrix involved) is already enough to learn a simple pronoun-to-subject coreference relationship from a single supervised example. This notebook follows the same self-attention pattern demonstrated in the "M9 Unit 6 Attention Mechanisms" class notebook, adapted to a different sentence and target relationship.